# BaginCam - uji super-resolusi burst RAW (algoritma Super Res Zoom Google)

Implementasi terbuka: [Jamy-L/Handheld-Multi-Frame-Super-Resolution](https://github.com/Jamy-L/Handheld-Multi-Frame-Super-Resolution) (MIT).

**Sebelum mulai:** menu *Runtime > Change runtime type > T4 GPU*, lalu jalankan sel dari atas ke bawah.
File yang diunggah tersimpan di server Google selama sesi ini saja.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Pasang program

In [ ]:
%cd /content
!rm -rf hmfsr && git clone -q --depth 1 https://github.com/Jamy-L/Handheld-Multi-Frame-Super-Resolution hmfsr
%cd /content/hmfsr
# Colab memakai CUDA 12 -> ganti paket numba-cuda cu13 ke cu12 bila ada
!sed -i 's/numba-cuda\[cu13\]/numba-cuda[cu12]/' pyproject.toml
!pip -q install . 2>&1 | tail -3
!handheld --help | head -40


## 2. Unggah burst (file zip berisi 20 DNG)
Pilih `burst_tripod.zip` atau `burst_tangan.zip` dari folder `data\bursts\colab` di PC.

In [ ]:
from google.colab import files
import zipfile, os, shutil
up = files.upload()
name = next(iter(up))
shutil.rmtree('/content/burst', ignore_errors=True)
os.makedirs('/content/burst')
zipfile.ZipFile(name).extractall('/content/burst')
print(len(os.listdir('/content/burst')), 'frame dari', name)


## 3. Jalankan (skala 2x)
Koreksi noise dimatikan dulu karena butuh tabel noise khusus kamera. Kalau sel ini gagal, salin pesan galatnya.

In [ ]:
%cd /content/hmfsr
!handheld --impath /content/burst --outpath /content/hasil_2x.png --scale 2 --robustness.no-noise-correction


## 4. Potong bagian tengah (bidang zoom 75x) dan unduh

In [ ]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
im = Image.open('/content/hasil_2x.png')
print('ukuran hasil', im.size)
w, h = im.size
crop = 15 / 1.25            # zoom 75x di tele 5x = crop 15x; diberi sedikit margin
cw, ch = int(w / crop), int(h / crop)
box = ((w - cw) // 2, (h - ch) // 2, (w + cw) // 2, (h + ch) // 2)
c = im.crop(box)
c.save('/content/tengah_75x.png')
c.resize((c.width * 4, c.height * 4), Image.LANCZOS).save('/content/tengah_75x_besar.png')
display(c.resize((c.width * 3, c.height * 3), Image.LANCZOS))


In [ ]:
from google.colab import files
files.download('/content/tengah_75x.png')
files.download('/content/tengah_75x_besar.png')


Simpan kedua file hasil unduhan ke `D:\project\ios\BaginCam\data\output\` lalu beri tahu Claude untuk dibandingkan dengan hasil Apple.